# M1 / M3 Muscarinic Receptor Leakage Analysis

In [ ]:
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
import scipy.stats
import matplotlib.pyplot as plt

from benchmark_data_leakage.chembl_requester import ChEMBLRequester
from benchmark_data_leakage.utils import find_repo_root

repo_root = find_repo_root()
fig_dir = repo_root / "figures"
fig_dir.mkdir(exist_ok=True)

## 1. Query ChEMBL

In [ ]:
with open(repo_root / "chembl_db.yaml") as f:
    chembl_db = yaml.safe_load(f)

req = ChEMBLRequester(**chembl_db)

ASSAY_M1 = "CHEMBL748282"
ASSAY_M3 = "CHEMBL745650"
focus_assays = (ASSAY_M1, ASSAY_M3)

In [ ]:
query = """
SELECT
    a2.chembl_id  AS assay_id,
    md.chembl_id  AS ligand_id,
    td.chembl_id  AS target_id,
    td.pref_name  AS target_name,
    cs.accession  AS uniprot_id,
    a.standard_type       AS measurement_type,
    a.standard_relation   AS measurement_relation,
    a.pchembl_value,
    a.standard_value,
    a.standard_units,
    a.activity_comment,
    a.data_validity_comment,
    a.potential_duplicate
FROM activities a
JOIN assays a2           ON a2.assay_id  = a.assay_id
JOIN molecule_dictionary md ON md.molregno = a.molregno
JOIN target_dictionary td   ON a2.tid      = td.tid
JOIN target_components tc   ON td.tid      = tc.tid
JOIN component_sequences cs ON tc.component_id = cs.component_id
WHERE a2.chembl_id IN %s
"""
req.cur.execute(query, (focus_assays,))
rows = req.cur.fetchall()
cols = [
    "assay_id", "ligand_id", "target_id", "target_name", "uniprot_id",
    "measurement_type", "measurement_relation", "pchembl_value",
    "standard_value", "standard_units", "activity_comment",
    "data_validity_comment", "potential_duplicate",
]
df_raw = pd.DataFrame(rows, columns=cols)
df_raw["pchembl_value"] = df_raw["pchembl_value"].astype(float)
df_raw

## 2. Validate shared compounds

In [ ]:
# Target metadata
print("Target info per assay:")
print(
    df_raw[["assay_id", "target_id", "target_name", "uniprot_id"]]
    .drop_duplicates()
    .to_string(index=False)
)
print()

# Per-assay compound counts
for assay_id, grp in df_raw.groupby("assay_id"):
    target = grp["target_name"].iloc[0]
    print(f"{assay_id} ({target}): {grp['ligand_id'].nunique()} unique compounds")

# Shared compounds
ligs_m1 = set(df_raw.loc[df_raw["assay_id"] == ASSAY_M1, "ligand_id"])
ligs_m3 = set(df_raw.loc[df_raw["assay_id"] == ASSAY_M3, "ligand_id"])
shared = ligs_m1 & ligs_m3
print(f"\nShared compounds: {len(shared)}")

In [ ]:
# Pivot to wide format — keep only shared compounds (inner join via dropna)
df_pivot = (
    df_raw[["assay_id", "ligand_id", "pchembl_value"]]
    .pivot(index="ligand_id", columns="assay_id", values="pchembl_value")
    .dropna()
)
print(f"Compounds with measurements in both assays: {len(df_pivot)}")
df_pivot

## 3. Scatter plot: M1 vs M3 pIC50

In [ ]:
label_m1 = df_raw.loc[df_raw["assay_id"] == ASSAY_M1, "target_name"].iloc[0]
label_m3 = df_raw.loc[df_raw["assay_id"] == ASSAY_M3, "target_name"].iloc[0]

x = df_pivot[ASSAY_M1].values  # M1 on x-axis
y = df_pivot[ASSAY_M3].values  # M3 on y-axis

corr, _ = scipy.stats.pearsonr(x, y)

slope, intercept, *_ = scipy.stats.linregress(x, y)
x_fit = np.linspace(x.min() - 0.15, x.max() + 0.15, 200)
y_fit = slope * x_fit + intercept

# 95% CI band
n = len(x)
x_mean = x.mean()
se_fit = (
    np.sqrt(np.sum((y - (slope * x + intercept)) ** 2) / (n - 2))
    * np.sqrt(1 / n + (x_fit - x_mean) ** 2 / np.sum((x - x_mean) ** 2))
)
t_crit = scipy.stats.t.ppf(0.975, n - 2)

fig, ax = plt.subplots(figsize=(4.5, 4.5))

ax.fill_between(
    x_fit, y_fit - t_crit * se_fit, y_fit + t_crit * se_fit,
    color="#3a6ea5", alpha=0.12, zorder=1,
)
ax.plot(x_fit, y_fit, color="#3a6ea5", linewidth=1.5, alpha=0.75, zorder=2)
ax.scatter(x, y, s=55, color="#1a3a5c", alpha=0.9, zorder=3, linewidths=0)

ax.text(
    0.06, 0.91, f"$r = {corr:.2f}$",
    transform=ax.transAxes, fontsize=12,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=2),
)

ax.set_xlabel(f"{label_m1} [pIC$_{{50}}$]", fontsize=11)
ax.set_ylabel(f"{label_m3} [pIC$_{{50}}$]", fontsize=11)
ax.tick_params(labelsize=9)
ax.grid(visible=True, which="major", linestyle=":", linewidth=0.5, color="#cccccc")
ax.set_axisbelow(True)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(fig_dir / "m1_m3_corrplot.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"n={n}, Pearson r={corr:.3f}")